# Validate Strategy

Stress-test a strategy before committing it to paper trading.  Six
PnL-based checks aggregated into a single go/no-go verdict:

1. **Plateau detection** — best params on a robust plateau or fragile spike?
2. **Walk-forward analysis** — do optimized params survive on unseen data?
3. **Bootstrap PnL CI** — how much depends on a few lucky trades?
4. **Rolling performance** — consistent across time windows?
5. **Fee sensitivity** — how much margin until breakeven?
6. **Regime breakdown** — clear edge in trending vs ranging markets?

Sharpe / Sortino / returns-based stats are deliberately absent — see
`docs/ANALYZER_RETURNS_CAVEAT.md`.

**Prerequisites:** Run a parameter sweep first (any
`notebooks/backtest/*.ipynb`) so a v2 sweep parquet exists at
`data/sweeps/{strategy}_{instrument}_{interval}.parquet`.

**Per-strategy switching:** edit the `STRATEGY` selector in cell 1.1 —
everything else is strategy-agnostic.


## 1. Setup


### 1.1 Imports & shared config

All editable knobs live here — strategy selector, sweep filters,
walk-forward fold sizes, bootstrap iterations, snapshot toggles.


In [ ]:
# ── Imports ────────────────────────────────────────────────────
from decimal import Decimal

import numpy as np
import pandas as pd
from IPython.display import display
from nautilus_trader.model.identifiers import Venue

from src.backtesting import (
    make_engine,
    performance_by_regime,
    performance_by_year,
    rolling_performance,
    run_fee_sweep,
    run_walk_forward,
    tag_regimes,
)
from src.backtesting.metrics import (
    bootstrap_max_drawdown,
    bootstrap_total_pnl,
)
from src.config.settings import get_settings
from src.core import bar_type_str, get_venue_config

# Add notebooks/ to sys.path so charts.py + utils.py imports resolve.
import sys
sys.path.insert(0, str(__import__("pathlib").Path(".").resolve()))

from charts import (
    plot_bootstrap_drawdown,
    plot_bootstrap_pnl,
    plot_equity_curve,
    plot_fee_sensitivity,
    plot_ma_cross,
    plot_pnl_heatmap,
    plot_regime_overlay,
    plot_rolling_pnl,
    plot_trade_distributions,
    plot_walkforward_oos_equity,
    plot_yearly_breakdown,
    print_yearly_breakdown,
)
from utils import (
    load_backtest_data,
    load_sweeps_filtered,
    make_instrument_id,
    print_validation_verdict,
    save_notebook_snapshot,
)

# Notebook-private helpers (extracted to keep cells short).  These
# live in notebooks/_validate_helpers.py — see that module for the
# strategy registry and per-cell helper functions.
from _validate_helpers import (
    collapse_to_grid,
    enrich_regime_with_wilson,
    get_param_grid,
    make_strategy_factory,
    parse_pnl,
    plateau_scores,
    short_params_tag,
)


# ─────────────────────────────────────────────────────────────────
# Editable config — strategy + instrument + run parameters
# ─────────────────────────────────────────────────────────────────
# STRATEGY is the registry key in _validate_helpers.STRATEGIES.
# The value stored in the sweep parquet's `_strategy` column also
# encodes the exec venue — see STRATEGY_FULL in derived values below.
STRATEGY         = "MACross-EMA"
# Defaults from src/config/settings.py (single source of truth across
# backtest, sandbox, live, and research).  Override per-session below.
settings         = get_settings()

DATA_SOURCE      = settings.data_source       # default: BINANCE_PERP
EXEC_VENUE       = settings.exec_venue        # default: HYPERLIQUID_PERP
ASSET            = "BTC"                       # research-specific (vary per run)
BAR_INTERVAL     = settings.bar_interval       # default: 4h

STARTING_CAPITAL = settings.starting_capital   # default: 1000
TRADE_SIZE       = int(settings.trade_notional)  # default: 2000
LEVERAGE         = settings.leverage            # default: 20

# Sweep filters
FILTER_LIQUIDATED = True   # drop liquidated rows from plateau analysis
FILTER_SPOTLIGHT  = True   # drop _kind == "spotlight" rows

# ─────────────────────────────────────────────────────────────────
# Override the auto-pick (optional)
# ─────────────────────────────────────────────────────────────────
# By default the load-sweep cell picks the row with the highest
# total_pnl_pct from the filtered sweep.  Set OVERRIDE_PARAMS to
# validate a SPECIFIC combo instead — e.g. the cross-sweep robust
# pick from compare_sweeps that doesn't match the per-sweep best.
#
# The override must match exactly one row in the loaded sweep; if
# no match is found, load-sweep raises RuntimeError with the row
# count for context.  Keys must be valid grid params for the
# selected STRATEGY (see _validate_helpers.STRATEGIES).
#
# Examples:
#   OVERRIDE_PARAMS = {"fast": 10, "slow": 20}            # MA crossovers
#   OVERRIDE_PARAMS = {"bb_period": 20, "bb_std": 2.0}    # BBMeanRev
#   OVERRIDE_PARAMS = None                                # auto-pick (default)
OVERRIDE_PARAMS: dict | None = None

# Walk-forward + bootstrap parameters
TRAIN_PCT        = 0.50      # fraction of bars used for training each fold
TEST_PCT         = 0.125     # fraction used for OOS test  (→ ~4 folds)
N_RESAMPLES      = 10_000    # bootstrap iterations
RANDOM_SEED      = 42

# Save behavior
SAVE_ON_RUN_ALL  = True


# ─────────────────────────────────────────────────────────────────
# Derived values — composed from the editable config above
# ─────────────────────────────────────────────────────────────────
DATA_CFG       = get_venue_config(DATA_SOURCE)
VENUE_CFG      = get_venue_config(EXEC_VENUE)
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, DATA_SOURCE)
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(DATA_CFG.nt_venue)

# STRATEGY_FULL is what `run_sweep` writes to the `_strategy` column
# in the parquet.  Validate filters sweeps against this, not STRATEGY.
STRATEGY_FULL  = f"{STRATEGY}-{EXEC_VENUE}"

CATALOG_PATH = "../data/catalog"

# RESULT_NAME — matches the backtest filename convention:
#   {prefix}_{strategy}_{ASSET}_{EXEC_VENUE}_{interval}[_{params}]
# Examples:
#   validate_MACross-EMA_BTC_HYPERLIQUID_PERP_1d            (auto-pick)
#   validate_MACross-EMA_BTC_HYPERLIQUID_PERP_1d_f10_s20    (override)
# short_params_tag compresses keys: fast→f, slow→s, bb_period→bp,
# bb_std→bs, dc_period→dp (first letter of each underscore-separated
# word).  Backtest convention: f10_s40 not fast10_slow40.
_OVERRIDE_TAG = (
    "_" + short_params_tag(OVERRIDE_PARAMS) if OVERRIDE_PARAMS else ""
)
RESULT_NAME  = f"validate_{STRATEGY}_{ASSET}_{EXEC_VENUE}_{BAR_INTERVAL}{_OVERRIDE_TAG}"

# Strategy factory + param grid resolved from the registry.  The
# factory matches run_sweep / run_walk_forward's expected signature:
# ``factory(engine, params_dict)``.
strategy_factory = make_strategy_factory(
    STRATEGY, INSTRUMENT_ID, BAR_TYPE_STR, TRADE_NOTIONAL,
)
PARAM_COMBOS, ROW_PARAM, COL_PARAM = get_param_grid(STRATEGY)


print(f"Strategy : {STRATEGY}  (sweep _strategy: {STRATEGY_FULL})")
print(f"Asset    : {INSTRUMENT_ID}")
print(f"Interval : {BAR_INTERVAL}")
print(f"Combos   : {len(PARAM_COMBOS)}")
if OVERRIDE_PARAMS is not None:
    override_str = ", ".join(f"{k}={v}" for k, v in OVERRIDE_PARAMS.items())
    print(f"Override : {override_str}  (auto-pick disabled)")
print("Imports OK")


### 1.2 Load data + sweep

Loads the OHLCV bars from the catalog and the matching v2 sweep
parquet (filtered through `load_sweeps_filtered`).  Picks the best
combo by `total_pnl_pct` from the filtered sweep.


In [ ]:
# ── Bars + instrument ───────────────────────────────────────────
# load_backtest_data applies the exec-venue fee overrides via
# venue_config — no separate with_venue_config() call needed.  Leverage
# flows directly into make_engine / run_walk_forward / run_fee_sweep
# via the leverage= kwarg, not the instrument.
instrument, bars = load_backtest_data(
    catalog_path=CATALOG_PATH,
    instrument_id=INSTRUMENT_ID,
    bar_type_str=BAR_TYPE_STR,
    venue_config=VENUE_CFG,
)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Bars       : {len(bars):,}")
print(f"Range      : {pd.Timestamp(bars[0].ts_event,  unit='ns', tz='UTC'):%Y-%m-%d}")
print(f"           → {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC'):%Y-%m-%d}")
print(f"Leverage   : {LEVERAGE}x  (applied at engine build time)")
print()

# ── Sweep parquet via v2 loader ─────────────────────────────────
# Filter against STRATEGY_FULL ("<base>-<EXEC_VENUE>") because that's
# what run_sweep writes to the parquet's `_strategy` column.
sweeps = load_sweeps_filtered(
    strategy=STRATEGY_FULL,
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    filter_liquidated=FILTER_LIQUIDATED,
    filter_spotlight=FILTER_SPOTLIGHT,
)

# Also load the unfiltered grid so the plateau cell can report the
# true survival rate (the filtered sweep above already drops liquidated
# combos — without the raw count, the plateau "100% profitable" verdict
# would silently exclude the ~30% of grid that blew up).
sweeps_raw = load_sweeps_filtered(
    strategy=STRATEGY_FULL,
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    filter_liquidated=False,
    filter_spotlight=True,   # spotlight is off-grid by design — still drop
)

if not sweeps:
    raise RuntimeError(
        f"No sweep parquet matches strategy={STRATEGY_FULL!r} "
        f"instrument={INSTRUMENT_ID!r} interval={BAR_INTERVAL!r}.  "
        "Run the corresponding backtest notebook first."
    )

# Expect exactly one match given the strict filter.
sweep_label, sweep_df = next(iter(sweeps.items()))
sweep_df_raw = next(iter(sweeps_raw.values()))
print(f"Loaded sweep: {sweep_label}  ({len(sweep_df)} survivors / {len(sweep_df_raw)} grid)")

# ── Pick the combo to validate ──────────────────────────────────
# Default: row with the highest total_pnl_pct (per-sweep "best").
# Override: a specific user-supplied combo via OVERRIDE_PARAMS in
# cell 1.1 — used to validate cross-sweep robust picks that aren't
# the per-sweep best.
if OVERRIDE_PARAMS is None:
    best_idx = sweep_df["total_pnl_pct"].idxmax()
else:
    # Build a row-mask matching every override key/value exactly.
    mask = pd.Series(data=True, index=sweep_df.index)
    for k, v in OVERRIDE_PARAMS.items():
        if k not in sweep_df.columns:
            raise RuntimeError(
                f"OVERRIDE_PARAMS key {k!r} is not a column in the loaded "
                f"sweep (columns: {sorted(sweep_df.columns)}).",
            )
        mask &= sweep_df[k] == v
    matching = sweep_df[mask]
    if matching.empty:
        param_str = ", ".join(f"{k}={v}" for k, v in OVERRIDE_PARAMS.items())
        raise RuntimeError(
            f"OVERRIDE_PARAMS {{{param_str}}} doesn't match any row in the "
            f"loaded sweep ({len(sweep_df)} rows after filters).  "
            "Either the combo wasn't part of the sweep grid, or it was "
            "filtered out (liquidated / spotlight).",
        )
    if len(matching) > 1:
        # Sweep with extra param dimensions (e.g. MACrossATR has
        # atr_sl/atr_tp on top of fast/slow) — pick the row with the
        # best total_pnl among matching rows.  Surface the choice so
        # the user knows we collapsed.
        print(
            f"⚠️ OVERRIDE_PARAMS matched {len(matching)} rows — picking the "
            "row with the highest total_pnl among them.",
        )
        best_idx = matching["total_pnl"].idxmax()
    else:
        best_idx = matching.index[0]

best_row = sweep_df.loc[best_idx]
param_keys = list(PARAM_COMBOS[0].keys())
best_params = {
    k: (best_row[k].item() if hasattr(best_row[k], "item") else best_row[k])
    for k in param_keys
}
best_pnl = float(best_row["total_pnl"])
param_str = ", ".join(f"{k}={v}" for k, v in best_params.items())
pick_source = "OVERRIDE" if OVERRIDE_PARAMS is not None else "auto-pick"
print(
    f"Validating ({pick_source}): {param_str}   "
    f"PnL={best_pnl:,.2f}  PnL%={best_row['total_pnl_pct']:.2f}",
)


## 2. Plateau detection


### 2.1 Plateau scoring

For each combo, count how many of its 3×3 grid neighbours (plus self)
are also profitable.  High = robust plateau.  Low = fragile spike.
Spotlight rows are excluded by `load_sweeps_filtered` upstream.


In [ ]:
# ── Survival accounting (no survivor bias) ─────────────────────
# The post-filter sweep_df only contains combos that DIDN'T liquidate.
# Surface that survival rate explicitly so the plateau verdict can't
# silently hide the blow-up rate.
n_grid       = len(sweep_df_raw)
n_liquidated = int(sweep_df_raw["liquidated"].sum()) if "liquidated" in sweep_df_raw.columns else 0
n_survivors  = n_grid - n_liquidated
survival_rate = n_survivors / n_grid if n_grid else 0.0

print(f"Grid coverage:")
print(f"  Total grid combos: {n_grid}")
print(f"  Liquidated:        {n_liquidated} ({(1 - survival_rate) * 100:.0f}% of grid blew up)")
print(f"  Survivors:         {n_survivors} ({survival_rate * 100:.0f}%)")
if n_liquidated > 0:
    print(f"  ⚠️ Plateau analysis below operates on SURVIVORS only —")
    print(f"     adjacent params that liquidated are not in the heatmap.")
print()

# ── Score the survivors ────────────────────────────────────────
# collapse_to_grid is a no-op for sweeps with unique (row, col)
# pairs.  For sweeps with extra param dimensions (e.g. MACrossATR
# adds atr_sl/atr_tp on top of fast/slow) it picks the best
# total_pnl per cell so the heatmap is well-defined.
grid_df = collapse_to_grid(sweep_df, ROW_PARAM, COL_PARAM)
scored  = plateau_scores(grid_df, ROW_PARAM, COL_PARAM)

# Locate best combo's score in the collapsed grid.
best_grid_idx = scored.loc[
    (scored[ROW_PARAM] == best_row[ROW_PARAM])
    & (scored[COL_PARAM] == best_row[COL_PARAM])
].index[0]
best_score = float(scored.loc[best_grid_idx, "neighbour_score"])
best_avg   = float(scored.loc[best_grid_idx, "neighbour_avg"])

n_profitable   = int(scored["profitable"].sum())
n_total        = int(len(scored))
pct_profitable = n_profitable / n_total * 100
median_score   = float(scored.loc[scored["profitable"], "neighbour_score"].median())

print(f"Survivor plateau:")
print(f"  {n_profitable}/{n_total} survivors profitable ({pct_profitable:.0f}%)")
print(f"  Median neighbour score (profitable survivors): {median_score:.2f}")
print()
print(f"Best combo ({COL_PARAM}={best_row[COL_PARAM]}, {ROW_PARAM}={best_row[ROW_PARAM]}):")
print(f"  PnL:             {best_pnl:,.2f}")
print(f"  Neighbour score: {best_score:.2f}  (1.0 = all neighbours profitable)")
print(f"  Neighbour avg:   {best_avg:,.2f}")
print()
if best_score >= 0.8:
    print("✅ PLATEAU — best params sit in a robust profitable region (among survivors).")
elif best_score >= 0.5:
    print("⚠️ RIDGE — best params are on the edge of profitability. Moderate risk.")
else:
    print("🚩 SPIKE — best params are an isolated peak. High overfit risk.")


### 2.2 Heatmap with best combo marked

Side-by-side PnL heatmap (bright green = winners, red = losers) and
neighbour-score heatmap (bright = robust).  The best combo is marked
with a star on both.  Spotlight rows are filtered defensively.


In [ ]:
# Heatmap shows the FULL grid (including liquidated combos), with
# liquidated cells greyed-out + underlined via the `flag_col` overlay.
# This avoids the "sparse heatmap" problem of plotting only survivors —
# without the flag, empty cells look identical to "no data" instead of
# "this combo blew up".
#
# Plateau scoring above still uses survivors-only (grid_df) — that's
# correct for the plateau verdict.  The heatmap rendering is the only
# place where seeing the full landscape is more honest than hiding it.
grid_df_raw = collapse_to_grid(sweep_df_raw, ROW_PARAM, COL_PARAM)

plot_pnl_heatmap(
    grid_df_raw,
    row_col=ROW_PARAM, col_col=COL_PARAM,
    value_col="total_pnl_pct",
    row_label=f"{ROW_PARAM.title()} Period",
    col_label=f"{COL_PARAM.title()} Period",
    title=f"PnL% — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}",
    fmt=".1f",
    flag_col="liquidated",
    flag_label="liquidated",
    exclude_kinds=("spotlight",),
)


## 3. Walk-forward analysis


### 3.1 Run walk-forward

Runs the full param sweep on each training window, picks the best by
total PnL, then tests that combo on the next OOS window.  Expensive —
one full sweep per fold.  Defaults: 50% train, 12.5% test → ~4 folds.

> **Note on `OVERRIDE_PARAMS`:** walk-forward re-optimizes across the
> full grid for each fold and picks the best per fold — it doesn't
> consult `OVERRIDE_PARAMS`.  So WF results + the param-stability
> verdict line are **per-instrument signals**, not per-combo: same
> output whether you ran auto-pick or override.  The sections that
> *do* differ between auto and override are 4.1 (single-config
> backtest), 5.x (bootstrap), 6 (rolling), and the verdict.


In [ ]:
wf_results = run_walk_forward(
    venue=VENUE,
    instrument=instrument,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=PARAM_COMBOS,
    strategy_factory=strategy_factory,
    train_pct=TRAIN_PCT,
    test_pct=TEST_PCT,
    select_by="total_pnl",
    leverage=LEVERAGE,
    venue_config=VENUE_CFG,
)
print(f"Folds completed: {len(wf_results)}")


### 3.2 Per-fold results table

OOS PnL alongside the best in-sample combo per fold.  If the best
combo keeps drifting between folds, the optimiser is curve-fitting to
each window rather than finding a real signal.


In [ ]:
if not wf_results.empty:
    display_cols = [c for c in wf_results.columns if c not in (
        "train_start", "train_end", "test_start", "test_end",
        "train_bars", "test_bars", "oos_error",
    )]
    print("=== Walk-forward fold results ===")
    display(wf_results[display_cols])
else:
    print("Walk-forward produced no folds — increase data length.")


### 3.3 In-sample vs OOS chart

Side-by-side bar chart of OOS PnL per fold (left) and in-sample vs
OOS comparison (right).  A healthy strategy keeps OOS PnL positive
even if it drops below in-sample (some haircut is normal — see the 30-
40% rule of thumb).


In [ ]:
if not wf_results.empty:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: OOS PnL per fold (signed colors)
    ax = axes[0]
    colors = ["#26a69a" if p > 0 else "#ef5350" for p in wf_results["oos_pnl"]]
    ax.bar(wf_results["fold"].astype(str), wf_results["oos_pnl"], color=colors)
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("Fold")
    ax.set_ylabel(f"Out-of-sample PnL ({CURRENCY})")
    ax.set_title("OOS PnL by fold")

    # Right: In-sample vs OOS bars
    ax = axes[1]
    x = np.arange(len(wf_results))
    w = 0.35
    ax.bar(x - w/2, wf_results["in_sample_pnl"], w, label="In-sample",   color="#2196f3", alpha=0.7)
    ax.bar(x + w/2, wf_results["oos_pnl"],       w, label="Out-of-sample", color="#f5c518", alpha=0.7)
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(wf_results["fold"].astype(str))
    ax.set_xlabel("Fold")
    ax.set_ylabel(f"PnL ({CURRENCY})")
    ax.set_title("In-sample vs out-of-sample")
    ax.legend()

    for ax_ in axes:
        ax_.set_facecolor("#131722")
        ax_.tick_params(colors="#d1d4dc")
        ax_.xaxis.label.set_color("#d1d4dc")
        ax_.yaxis.label.set_color("#d1d4dc")
        ax_.title.set_color("#d1d4dc")
    fig.patch.set_facecolor("#131722")

    plt.tight_layout()
    plt.show()


### 3.4 Stitched OOS equity curve

The canonical walk-forward chart — "what if I had actually traded the
WF picks live?".  Each fold contributes a line segment from
(test_start, prior_cum) to (test_end, prior_cum + oos_pnl);
slope shows that fold's per-day OOS rate.  Final cumulative number
in the headline tells you the OOS return you would have earned
trading the optimiser's picks across all folds.


In [ ]:
if not wf_results.empty:
    plot_walkforward_oos_equity(
        wf_results,
        title=f"Walk-forward stitched OOS equity — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}",
        currency=str(CURRENCY),
    )


## 4. Single-config performance


### 4.1 Run a single backtest at best params

Re-runs the full backtest with the best combo to extract per-trade
PnL plus the orders / account / positions reports the cells below
consume.  Override `VALIDATE_PARAMS` here if walk-forward suggested
a different (more robust) combo than the one that maxed total PnL.


In [ ]:
VALIDATE_PARAMS = dict(best_params)
print(f"Validating: {', '.join(f'{k}={v}' for k, v in VALIDATE_PARAMS.items())}")

eng = make_engine(
    VENUE, instrument, bars, STARTING_CAPITAL,
    leverage=LEVERAGE,
    venue_config=VENUE_CFG,
)
strategy_factory(eng, VALIDATE_PARAMS)
eng.run()

# Capture all reports BEFORE disposal — downstream cells (price chart,
# equity curve, per-year, regime) need them and engine.dispose()
# invalidates the cache.
positions_report = eng.trader.generate_positions_report()
fills_report     = eng.trader.generate_order_fills_report()
account_report   = eng.trader.generate_account_report(VENUE)
positions = eng.cache.position_snapshots() + eng.cache.positions()
eng.dispose()

if positions_report.empty:
    trade_pnls: list[float] = []
    print("No positions — bootstrap will be skipped.")
else:
    series = positions_report["realized_pnl"].apply(parse_pnl).dropna()
    trade_pnls = [float(x) for x in series.values]

    n_trades  = len(trade_pnls)
    total_pnl = sum(trade_pnls)
    winners   = sum(1 for x in trade_pnls if x > 0)
    losers    = sum(1 for x in trade_pnls if x < 0)
    print(f"Trades: {n_trades}  (W:{winners} / L:{losers})")
    print(f"Total PnL: {total_pnl:,.2f}")
    if n_trades >= 1:
        print(f"Mean per trade  : {total_pnl / n_trades:,.2f}")
        print(f"Median per trade: {float(np.median(trade_pnls)):,.2f}")


### 4.2 Price chart with trade markers

The single most important chart for any backtest — *where* did the
strategy trade and how did those entries align with price action?
Plotly figure with candlesticks + EMA overlays + buy/sell markers.


In [ ]:
if not fills_report.empty:
    fig = plot_ma_cross(
        bars, fills_report,
        fast_period=VALIDATE_PARAMS["fast"],
        slow_period=VALIDATE_PARAMS["slow"],
        ma_type="EMA",
        instrument_label=str(INSTRUMENT_ID),
        bar_label=BAR_INTERVAL,
        height=600,
    )
    fig.show()
else:
    print("No fills — nothing to plot.")


### 4.3 Equity & drawdown (event-time)

Cumulative balance + running peak + drawdown on a single chart.
Event-time means the line steps at fills — between events the balance
is held flat (intra-bar drift on open positions is invisible).


In [ ]:
plot_equity_curve(
    account_report,
    f"Equity & drawdown — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}",
    currency=str(CURRENCY),
)


### 4.4 Per-year breakdown

Decomposes total PnL into calendar-year buckets — answers the most-
asked question on any 5+ year backtest: "did this work *only* in 2020-
2021 covid pump or also before/after?"  A trend-follower with all
profit in one year is fragile; consistent year-over-year wins are
what you want for paper trading.


In [ ]:
yearly_df = performance_by_year(positions, starting_capital=STARTING_CAPITAL)
print_yearly_breakdown(yearly_df, currency=str(CURRENCY))
plot_yearly_breakdown(
    yearly_df,
    title=f"Per-year PnL — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}",
    currency=str(CURRENCY),
)


### 4.5 Trade distributions

Three-panel view of the per-trade record: PnL distribution (winners
green / losers red), trade duration distribution, and top-trade
share (what % of total PnL came from the top N trades).  Reveals
whether the strategy's edge is broad-based or concentrated in a
handful of outliers — independent of the bootstrap resampling above.


In [ ]:
if positions:
    plot_trade_distributions(
        positions,
        title=f"Trade distributions — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}",
        currency=str(CURRENCY),
    )
else:
    print("No positions — nothing to plot.")


## 5. Bootstrap analysis


### 5.1 Bootstrap PnL confidence interval

Calls `bootstrap_total_pnl()` from `src.backtesting.metrics` to
resample per-trade PnL `N_RESAMPLES` times with replacement and
return the 5/25/50/75/95 percentiles.  P(profit) tells you the
fraction of resamples that ended positive.


In [ ]:
if len(trade_pnls) >= 5:
    boot = bootstrap_total_pnl(
        trade_pnls,
        n_iterations=N_RESAMPLES,
        seed=RANDOM_SEED,
    )

    # Compute prob_positive from the same RNG (the metrics helper
    # returns summary stats only).  This is a tight loop but small —
    # n_iterations samples of n_trades each.
    rng = np.random.default_rng(RANDOM_SEED)
    n   = len(trade_pnls)
    arr = np.asarray(trade_pnls, dtype=float)
    idx = rng.integers(0, n, size=(N_RESAMPLES, n))
    totals = arr[idx].sum(axis=1)
    prob_positive = float((totals > 0).mean() * 100)

    p5, p95 = boot["pct_5"], boot["pct_95"]

    print(f"Bootstrap: {N_RESAMPLES:,} resamples of {n} trades")
    print()
    print(f"  5th  : {p5:>10,.2f}  (worst realistic case)")
    print(f"  Med. : {boot['median']:>10,.2f}")
    print(f"  95th : {p95:>10,.2f}  (best realistic case)")
    print(f"  Actual : {boot['actual_total']:>10,.2f}")
    print(f"  P(profit): {prob_positive:.1f}%")
    print()
    print(f"  90% CI : [{p5:,.2f}, {p95:,.2f}]")
    print()
    # P0.2 — IID caveat.  bootstrap_total_pnl resamples per-trade PnLs
    # independently, which understates real-world drawdowns / overstates
    # confidence for any strategy with autocorrelation (trend-followers
    # cluster wins; mean-reverters cluster losses in chop).  Treat
    # P(profit) as an upper bound, not a true confidence.
    print("  ⚠️ Assumes IID trades — path dependence not captured.")
    print("    Treat P(profit) as an upper bound on true confidence.")
else:
    boot = None
    prob_positive = float("nan")
    p5 = p95 = float("nan")
    print(f"Only {len(trade_pnls)} trades — need at least 5 for meaningful bootstrap.")


### 5.2 Bootstrap PnL distribution chart

Histogram (Gaussian approximation overlaid with empirical
percentiles) — see `plot_bootstrap_pnl` for the caveats.


In [ ]:
if boot is not None:
    plot_bootstrap_pnl(
        boot,
        title=(
            f"Bootstrap PnL — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}"
        ),
        currency=str(CURRENCY),
    )


### 5.3 Bootstrap max-drawdown CI

Companion to the PnL bootstrap — resamples the per-trade PnL list and
records the worst peak-to-trough drawdown of each synthetic equity
curve.  Pros size positions to survive their *drawdown* CI, not their
PnL CI.  The ``actual`` line shows where the historical drawdown sits
in the resampled distribution.


In [ ]:
if len(trade_pnls) >= 5:
    boot_dd = bootstrap_max_drawdown(
        trade_pnls,
        n_iterations=N_RESAMPLES,
        seed=RANDOM_SEED,
    )
    print(f"Max-drawdown bootstrap: {N_RESAMPLES:,} resamples of {len(trade_pnls)} trades")
    print()
    # Drawdowns are negative; pct_5 = worst tail, pct_95 = least bad.
    print(f"  Worst 5%   : {boot_dd['pct_5']:>10,.2f}  (deep tail)")
    print(f"  Median     : {boot_dd['median']:>10,.2f}")
    print(f"  Best 95%   : {boot_dd['pct_95']:>10,.2f}  (shallow tail)")
    print(f"  Actual MDD : {boot_dd['actual_max_drawdown']:>10,.2f}")
    print()
    print(f"  Position-sizing question: can you survive {boot_dd['pct_5']:,.0f} "
          "before the strategy recovers?")
    print()
    print("  ⚠️ Same IID caveat as PnL bootstrap — real drawdowns")
    print("    cluster (e.g. choppy regime → run of stops), so the")
    print("    worst-5% here is a LOWER bound on realistic worst-case.")

    plot_bootstrap_drawdown(
        boot_dd,
        title=f"Bootstrap MDD — {STRATEGY} {INSTRUMENT_ID} {BAR_INTERVAL}",
        currency=str(CURRENCY),
    )
else:
    boot_dd = None
    print("Skipping max-drawdown bootstrap — need ≥5 trades.")


## 6. Rolling performance

90-day rolling-window PnL and trade count.  A strategy that's
profitable overall but loses money in 50% of windows is concentrated
into a few hot streaks — fragile.  Active windows (with at least
one trade) are reported separately from inactive ones (sparse
strategies have many no-trade windows that shouldn't count
against them).


In [ ]:
rolling_df = rolling_performance(positions, bars, window="90D")

if not rolling_df.empty:
    display(rolling_df)
    plot_rolling_pnl(rolling_df, currency=str(CURRENCY))

    # P0.3 — split active windows (had at least one closed trade) from
    # inactive windows (zero-PnL because no trade happened in that
    # 90-day stretch).  Counting zero-PnL windows as "not profitable"
    # makes the metric meaningless for sparse strategies (e.g. EMA
    # cross with 23 trades over 6.5 years → most windows are dead air).
    active     = rolling_df[rolling_df["pnl"] != 0.0]
    inactive   = rolling_df[rolling_df["pnl"] == 0.0]
    pos_active = int((active["pnl"] > 0).sum())
    n_active   = len(active)
    n_inactive = len(inactive)

    print(f"Rolling 90-day windows:")
    print(f"  Total windows    : {len(rolling_df)}")
    print(f"  Active (had trades): {n_active}")
    print(f"  Inactive (no trades): {n_inactive}")
    if n_active > 0:
        pct = pos_active / n_active * 100
        print(f"  Active windows profitable: {pos_active}/{n_active} ({pct:.0f}%)")
    else:
        print(f"  No active windows — strategy traded zero closed positions.")
else:
    print("No rolling-window results.")


## 7. Fee sensitivity

Re-runs the strategy at increasing fee levels and finds the breakeven
point.  Strong margin = profitable up to ≥7.5 bps.  Thin margin = you
lose your edge to a fee increase or worse fills.


In [ ]:
fee_df = run_fee_sweep(
    venue=VENUE,
    instrument=instrument,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    params=VALIDATE_PARAMS,
    strategy_factory=strategy_factory,
    leverage=LEVERAGE,
)
if not fee_df.empty:
    display(fee_df[[
        "fee_bps", "total_pnl", "total_pnl_pct", "pnl_per_trade", "breakeven",
    ]])
    plot_fee_sensitivity(fee_df, currency=str(CURRENCY))
else:
    print("Fee sweep produced no results.")


## 8. Regime breakdown

ADX-tagged regimes (TRENDING vs RANGING vs CHOP).  Per-regime stats
include Wilson 95% confidence intervals on win-rate so small-sample
regimes (n<10 trades) don't get reported with the same gravitas as
well-sampled ones.  A trend-follower that loses money in ranging
regimes is fine *if* it's net positive across the regime mix and
trending PnL > |ranging PnL|.


In [ ]:
regime_df   = tag_regimes(bars, method="adx")
regime_perf = performance_by_regime(positions, regime_df)
if not regime_perf.empty:
    # enrich_regime_with_wilson adds a `wr_ci` column with the
    # 95% Wilson interval on win-rate so small-sample regimes
    # (n<10 trades) get appropriately humble error bars.
    display(enrich_regime_with_wilson(regime_perf))
    plot_regime_overlay(regime_df)

    counts = regime_df["regime"].value_counts(normalize=True)
    print("Regime distribution (% of bars):")
    for regime, pct in counts.items():
        print(f"  {regime}: {pct*100:.1f}%")

    # Flag low-confidence regime stats explicitly.
    low_conf = regime_perf[regime_perf["num_positions"] < 10]
    if not low_conf.empty:
        print()
        print("⚠️ Low-confidence regimes (n<10 trades):")
        for _, row in low_conf.iterrows():
            print(f"  {row['regime']}: {int(row['num_positions'])} trades — treat win-rate / PF as indicative only")
else:
    print("No regime-perf rows.")


## 9. Go / no-go assessment

Aggregates the PnL-based checks above into one verdict via
`print_validation_verdict`.  All inputs are optional — pass `None`
for any check you didn't run.


In [ ]:
n_trades_for_boot = len(trade_pnls) if "trade_pnls" in dir() else None

# Verdict JSON lands next to the snapshot HTML so validate_all.ipynb
# can consolidate runs into a comparison matrix without parsing
# stdout.  RESULT_NAME tags the override (if any) so back-to-back
# auto-pick + override-pick runs don't overwrite each other.
verdict_json_path = (
    __import__("pathlib").Path("../reports/validate")
    / f"{RESULT_NAME}_verdict.json"
)

# Trailing semicolon suppresses Jupyter's auto-display of the returned
# verdict dict (which is already on disk as JSON and printed in
# formatted form above — no need to dump it twice).
print_validation_verdict(
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    params=VALIDATE_PARAMS,
    plateau_score=best_score,
    walkforward_results=wf_results,
    bootstrap_prob_positive=prob_positive if boot is not None else None,
    bootstrap_p5=p5 if boot is not None else None,
    bootstrap_p95=p95 if boot is not None else None,
    n_trades=n_trades_for_boot,
    rolling_results=rolling_df,
    fee_results=fee_df,
    regime_results=regime_perf,
    yearly_results=yearly_df,
    starting_capital=STARTING_CAPITAL,
    verdict_path=verdict_json_path,
    # When override is set, persist it explicitly in the verdict JSON
    # so validate_all can distinguish auto vs override rows without
    # filename-suffix sniffing.
    override_params=OVERRIDE_PARAMS,
);


## 10. Save & cleanup


### 10.1 Save snapshot

Writes the executed notebook + rendered HTML to
`reports/notebooks/validate/` and `reports/html/validate/`.  Honors
`SAVE_ON_RUN_ALL` from cell 1.1.


In [ ]:
# Trailing semicolon suppresses Jupyter's auto-display of the
# returned Path (the helper already prints "Saved -> ..." for both
# .ipynb and .html).
save_notebook_snapshot(
    "validate_strategy.ipynb", RESULT_NAME,
    save_on_run_all=SAVE_ON_RUN_ALL,
    category="validate",
);


### 10.2 Cleanup

The bootstrap engine was already disposed in 4.1.  Nothing else to
release.


In [ ]:
print("Done.")


## 11. Scratchpad

Ad-hoc analysis goes here.  Anything below this line is excluded from
the canonical layout.


In [ ]:
# Scratchpad — your ad-hoc analysis goes here.
